# มนุษย์ในวงจร: ประตูการดำเนินการล่วงหน้า การจัดระดับความเสี่ยง และการบันทึกตรวจสอบ

README สำหรับบทเรียนนี้แนะนำมนุษย์ในวงจรด้วยตัวอย่างสั้น ๆ ที่ถามผู้ใช้ว่า `APPROVE` หรือ `REJECT` หลังจากที่เอเจนต์ได้ตอบกลับแล้ว รูปแบบนั้นเป็นจุดเริ่มต้นที่ดี แต่การนำ HITL ไปใช้งานจริงมักต้องการสามส่วนเพิ่มเติม:

1. **ประตูการดำเนินการล่วงหน้า** ที่ทำงาน **ก่อน** เอเจนต์จะดำเนินการขั้นตอนที่มีความเสี่ยง เพื่อควบคุมค่าใช้จ่าย ความไม่ย้อนกลับได้ และความล่าช้า
2. **การจัดระดับความเสี่ยง** เพื่อให้การดำเนินการความเสี่ยงต่ำทำงานโดยอัตโนมัติ การดำเนินการความเสี่ยงปานกลางได้รับการอนุมัติเป็นกลุ่ม และการดำเนินการความเสี่ยงสูงเท่านั้นที่ต้องรอมนุษย์
3. **บันทึกการตรวจสอบและวงจรการแก้ไข** เพื่อให้ทุกการตัดสินใจที่ประตูถูกบันทึกเป็น JSONL และการปฏิเสธจะกระตุ้นเอเจนต์ใหม่ด้วยเหตุผลที่มีโครงสร้างแทนที่จะพิมพ์แค่ `Revising...`

โน้ตบุ๊กนี้สร้างแต่ละส่วนเหล่านี้บนพื้นฐานของ primitive เดียวกับใน `06-system-message-framework.ipynb` มันรันแบบครบวงจรในโหมด `DEMO_MODE = True` (ไม่ต้องใช้การป้อนข้อมูลแบบโต้ตอบ) หรือกับพร้อมท์ `input()` จริงเมื่อ `DEMO_MODE = False` หมายเหตุ: ใน DEMO_MODE การลองใหม่สำหรับเป้าหมายที่สามถูกเขียนสคริปต์ไว้เพื่อให้เห็นกลไกของวงจรอย่างครบถ้วน การจัดประเภทใหม่ที่ขับเคลื่อนโดยการแก้ไขจริงต้องใช้ `DEMO_MODE = False` และผู้ควบคุม

**อยู่นอกขอบเขต (จัดการในบทเรียนอื่น):** การพิสูจน์ตัวตนและการควบคุมการเข้าถึง (บทเรียน 06 README ภัยคุกคามที่ 2), ตัวกลางการเรียกใช้งานเครื่องมือ (บทเรียน 14 การเจาะลึก MAF), รูปแบบการถกเถียงของเอเจนต์หลายตัว


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## แบบที่ 1: ประตูป้องกันก่อนทำการ

ชิ้นส่วน HITL ใน README เรียกตัวแทนก่อน แล้วจึงขออนุมัติผลลัพธ์จากผู้ใช้ ซึ่งเป็นลำดับการทำงานแบบ **หลังดำเนินการ** ตัวแทนได้ทำงานไปแล้ว จึงเสียค่าใช้จ่ายในการเรียก LLM ไปแล้ว และผลข้างเคียงใด ๆ (เช่น การส่งอีเมล เขียนแถวในฐานข้อมูล แสดงความคิดเห็น) ก็เกิดขึ้นไปแล้ว

ลำดับการทำงานแบบ **ก่อนดำเนินการ** จะใส่ประตูกั้นก่อนที่ตัวแทนจะทำขั้นตอนที่มีความเสี่ยง ตัวแทนเสนอการกระทำ ประตูจะตัดสินใจว่าจะดำเนินการหรือไม่ และจะมีผลข้างเคียงก็ต่อเมื่อได้รับการอนุมัติเท่านั้น

| แง่มุม | การอนุมัติหลังดำเนินการ (ชิ้นส่วน README) | ประตูป้องกันก่อนดำเนินการ (สมุดบันทึกนี้) |
|---|---|---|
| การอนุมัติทำงานเมื่อใด? | หลังจากที่ตัวแทนได้สร้างผลลัพธ์แล้ว | ก่อนที่ผลข้างเคียงใด ๆ จะเกิดขึ้น |
| ค่าใช้จ่าย LLM เมื่อถูกปฏิเสธ | จ่ายไปแล้ว | จ่ายเฉพาะค่าข้อเสนอ ไม่ใช่การกระทำ |
| ผลข้างเคียงที่ไม่สามารถย้อนกลับได้ | เป็นไปได้ (การกระทำได้เกิดขึ้นแล้ว) | ป้องกันได้ |
| ความชัดเจนในการตรวจสอบ | การอนุมัติเป็นข้อความพิมพ์ | การอนุมัติเป็นบันทึก JSON พร้อมเวลาบันทึก การกระทำ เหตุผล |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## รูปแบบที่ 2: การจัดระดับความเสี่ยง

ไม่ใช่ทุกการกระทำที่จะต้องได้รับการอนุมัติจากมนุษย์ การตรวจสอบแบบอ่านอย่างเดียวกับ API สาธารณะมีความเสี่ยงที่แตกต่างจากการส่งอีเมลถึงลูกค้า การปฏิบัติทั้งสองแบบเหมือนกันทำให้เสียสมาธิผู้ปฏิบัติงานและทำให้เอเย่นต์ทำงานช้าลง

แบบจำลอง 3 ระดับง่ายๆ:

| ระดับ | ตัวอย่าง | ขั้นตอนการอนุมัติ |
|---|---|---|
| `ต่ำ` (อ่านอย่างเดียว) | ค้นหาฐานความรู้, ตรวจสอบตัวเลือกเที่ยวบิน, ดึงหน้าเว็บสาธารณะ | ดำเนินการอัตโนมัติ, บันทึกสำหรับตรวจสอบ |
| `กลาง` (การเปลี่ยนแปลงราคาถูก) | แคชผลลัพธ์, เพิ่มตัวนับ, กำหนดเตือนความจำ | ดำเนินการอัตโนมัติ, แต่ตรวจสอบรวมรายวัน |
| `สูง` (เผชิญหน้าภายนอกหรือไม่สามารถย้อนกลับได้) | ส่งอีเมล, คิดค่าบัตร, โพสต์ในช่องทางสาธารณะ | รอการอนุมัติจากมนุษย์ |

นี่คือการจัดระดับแบบหนึ่ง ระบบการผลิตมักใช้ระดับที่ละเอียดกว่านี้ (เช่น ระดับการอนุญาต IAM ของ AWS, ระดับการเข้าถึงตามบทบาท) เวอร์ชัน 3 ระดับด้านล่างเป็นเวอร์ชันที่เล็กที่สุดที่มีประโยชน์สำหรับเอเย่นต์ที่ผสมผสานการกระทำที่อ่านอย่างเดียวกับที่มีผลข้างเคียง

ตัวจัดหมวดหมู่ด้านล่างใช้วิธีการหาคำหลักเพื่อให้การสาธิตยังคงกำหนดได้และมีราคาถูก ในระบบการผลิต คุณจะเปลี่ยนเป็นตัวจัดหมวดหมู่ที่เรียนรู้ได้หรือเครื่องยนต์นโยบาย


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## รูปแบบที่ 3: บันทึกตรวจสอบและวงจรแก้ไข

`print("Response approved.")` ไม่ใช่บันทึกตรวจสอบ สำหรับความน่าเชื่อถือ ทุกการตัดสินใจที่เกตควรถูกบันทึกเป็นเหตุการณ์ที่มีโครงสร้างที่คุณสามารถนำมาสืบค้น ทีหลังเล่นซ้ำ หรือแนบในการทบทวนเหตุการณ์ได้

สองส่วน:

1. **JSONL แบบเพิ่มอย่างเดียว** แต่ละบรรทัดสำหรับการตัดสินใจหนึ่งครั้ง พร้อมเวลาประทับ, การกระทำ, ชั้น, การตัดสินใจ, เหตุผล ง่ายต่อการ grep ง่ายต่อการส่งไปยังที่จัดเก็บบันทึกจริงในภายหลัง
2. **วงจรแก้ไขเมื่อถูกปฏิเสธ** เมื่อเกตส่งกลับ `deny` เอเจนต์จะถามตัวเองซ้ำด้วยเหตุผลการปฏิเสธในบริบท เพื่อที่ข้อเสนอถัดไปจะหลีกเลี่ยงปัญหาได้


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## แหล่งข้อมูลเพิ่มเติม

โครงการสาธารณะอื่นๆ หลายโครงการได้นำรูปแบบ HITL เหล่านี้ไปประยุกต์ใช้งานในรูปแบบต่างๆ กัน เปรียบเทียบวิธีการต่างๆ เพื่อหาแนวทางที่เหมาะกับสแตกของคุณ:

- **LangChain** ตัวห่อเครื่องมือแบบ human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): ตัวห่อเครื่องมือที่แทรกการหยุดชั่วคราวเพื่อรอรับข้อมูลจากมนุษย์
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ได้ปรับโครงสร้างใหม่นี้): ใช้บทบาทตัวแทนเฉพาะสำหรับแทนตัวมนุษย์ในบทสนทนาหลายเอเยนต์
- **Microsoft Agent Framework (MAF)** middleware การเรียกใช้งานฟังก์ชัน ([docs](https://learn.microsoft.com/agent-framework/)): middleware ที่ทำงานรอบๆ การเรียกใช้งานเครื่องมือ/ฟังก์ชันทุกครั้ง เหมาะสำหรับตรรกะการควบคุมและกระบวนการอนุมัติ

โครงการแต่ละโครงการจัดการกับสามรูปแบบย่อยต่างกัน: LangChain ห่อเป็นเครื่องมือ, AutoGen ใช้บทบาทตัวแทน, และ Microsoft Agent Framework ใช้ middleware การเรียกใช้งานฟังก์ชัน อ่านการนำไปใช้งานสักหนึ่งหรือสองโครงการให้ครบถ้วนก่อนตัดสินใจเลือกออกแบบเอเยนต์ของคุณเอง


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
